# Whirlpool Galaxy Telescope Live-Stack Video for YouTube Shorts

This notebook creates a **vertical 9:16 telescope-style live-stacking video** for **M51, the Whirlpool Galaxy**, with its companion **NGC 5195** and a visible tidal bridge.

It is designed as a more dynamic version of the Andromeda example: noisy raw frames gradually turn into a cleaner stacked image, with HUD telemetry, reticles, scan sweep, cosmic-ray streaks, and a final “tidal bridge resolved” beat.

**Outputs:**
- `whirlpool_galaxy_shorts_preview.png`
- `whirlpool_galaxy_telescope_live_stack_shorts.mp4`

Video settings: **720×1280**, **18 FPS**, **16 seconds**. This is vertical YouTube Shorts format and longer than 15 seconds.


In [ ]:
# Uncomment in a fresh environment if needed:
# %pip install -U numpy pillow imageio imageio-ffmpeg


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import json
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import imageio.v2 as imageio

OUT_DIR = Path('.')
VIDEO_PATH = OUT_DIR / 'whirlpool_galaxy_telescope_live_stack_shorts.mp4'
PNG_PATH = OUT_DIR / 'whirlpool_galaxy_shorts_preview.png'
NOTEBOOK_PATH = OUT_DIR / 'Whirlpool_Galaxy_Telescope_Live_Stack_Shorts.ipynb'

# YouTube Shorts-friendly vertical 9:16 canvas. 720p keeps rendering fast.
W, H = 720, 1280
FPS = 18
SECONDS = 16
NFRAMES = FPS * SECONDS
SEED = 5195
rng = np.random.default_rng(SEED)


def robust_scale(arr, lo=0.25, hi=99.85, gamma=0.72):
    arr = np.asarray(arr, dtype=np.float32)
    a, b = np.percentile(arr, [lo, hi])
    scaled = np.clip((arr - a) / max(b - a, 1e-6), 0, 1)
    return scaled ** gamma


def smooth_noise(width=W, height=H, seed=SEED, low=90, scale=1.0):
    local = np.random.default_rng(seed)
    small = local.random((max(8, height // low), max(8, width // low))).astype(np.float32)
    im = Image.fromarray((small * 255).astype(np.uint8), 'L').resize((width, height), Image.Resampling.BICUBIC)
    im = im.filter(ImageFilter.GaussianBlur(radius=10))
    arr = np.asarray(im).astype(np.float32) / 255.0
    return scale * arr


def make_starfield(width=W, height=H, seed=SEED):
    local = np.random.default_rng(seed)
    img = np.zeros((height, width), dtype=np.float32)
    yy_full, xx_full = np.mgrid[0:height, 0:width]
    nstars = 2400
    xs = local.uniform(0, width, nstars)
    ys = local.uniform(0, height, nstars)
    flux = 0.05 + 1.45 * (1 - local.power(3.5, nstars)) ** 2
    sigma = local.uniform(0.35, 1.15, nstars)
    # A few brighter foreground stars for Shorts visual pop.
    bright_idx = local.choice(nstars, size=38, replace=False)
    flux[bright_idx] *= local.uniform(1.8, 3.4, len(bright_idx))
    sigma[bright_idx] *= local.uniform(1.0, 1.8, len(bright_idx))

    for x, y, f, s in zip(xs, ys, flux, sigma):
        r = int(max(2, math.ceil(4 * s)))
        x0, x1 = max(0, int(x)-r), min(width, int(x)+r+1)
        y0, y1 = max(0, int(y)-r), min(height, int(y)+r+1)
        xx = xx_full[y0:y1, x0:x1]
        yy = yy_full[y0:y1, x0:x1]
        img[y0:y1, x0:x1] += f * np.exp(-((xx-x)**2 + (yy-y)**2) / (2*s*s))
    return img


def add_gaussian(blob, x, y, amp, sx, sy=None):
    if sy is None:
        sy = sx
    height, width = blob.shape
    r = int(max(3, math.ceil(4 * max(sx, sy))))
    x0, x1 = max(0, int(x)-r), min(width, int(x)+r+1)
    y0, y1 = max(0, int(y)-r), min(height, int(y)+r+1)
    if x1 <= x0 or y1 <= y0:
        return
    yy, xx = np.mgrid[y0:y1, x0:x1]
    blob[y0:y1, x0:x1] += amp * np.exp(-(((xx-x)/sx)**2 + ((yy-y)/sy)**2) / 2)


def make_whirlpool_base(width=W, height=H, seed=SEED):
    local = np.random.default_rng(seed)
    y, x = np.mgrid[0:height, 0:width]

    # Main face-on spiral: M51 / Whirlpool Galaxy.
    cx, cy = width * 0.50, height * 0.52
    theta = np.deg2rad(-17)
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    scale = width * 0.265
    r = np.sqrt((xr/scale)**2 + (yr/scale)**2)
    phi = np.arctan2(yr, xr)

    disk = 0.95 * np.exp(-1.9 * r)
    core = 2.4 * np.exp(-((xr/(width*0.052))**2 + (yr/(width*0.052))**2))
    spiral_phase = np.sin(2.0 * phi + 5.2 * r - 0.55)
    arms = 1.30 * np.exp(-1.25*r) * np.exp(-(spiral_phase**2) / (0.030 + 0.030*r))
    outer_arms = 0.55 * np.exp(-0.7*r) * np.exp(-(np.sin(2.0*phi + 4.0*r + 1.2)**2) / (0.070 + 0.035*r))

    # Dust lanes with soft fractal variation.
    dust_noise = smooth_noise(width, height, seed+11, low=72, scale=1.0)
    dust_phase = np.exp(-(np.sin(2.0 * phi + 5.0*r + 0.15)**2) / (0.024 + 0.025*r)) * np.exp(-1.25*r)
    dust = 0.36 * dust_phase * (0.65 + 0.70*dust_noise)

    main = np.clip(disk + core + arms + outer_arms, 0, None) * np.clip(1 - dust, 0.28, 1.0)

    # Companion galaxy NGC 5195 and tidal bridge.
    c2x, c2y = width * 0.59, height * 0.30
    comp_x = (x - c2x)
    comp_y = (y - c2y)
    companion = 1.20 * np.exp(-((comp_x/(width*0.075))**2 + (comp_y/(width*0.058))**2))
    companion += 0.40 * np.exp(-np.sqrt((comp_x/(width*0.19))**2 + (comp_y/(width*0.16))**2))

    # Soft bridge between galaxies: distance to line segment.
    ax, ay = cx + width*0.08, cy - height*0.14
    bx, by = c2x - width*0.02, c2y + height*0.05
    vx, vy = bx - ax, by - ay
    t = np.clip(((x - ax)*vx + (y - ay)*vy) / max(vx*vx + vy*vy, 1e-6), 0, 1)
    px, py = ax + t*vx, ay + t*vy
    dline = np.sqrt((x-px)**2 + (y-py)**2)
    bridge = 0.50 * np.exp(-(dline/(width*0.035))**2) * np.exp(-0.8*(1-t))

    # Star-forming knots placed along spiral arms.
    knots_blue = np.zeros((height, width), dtype=np.float32)
    knots_red = np.zeros((height, width), dtype=np.float32)
    for branch in [0, math.pi]:
        for _ in range(110):
            rr = local.uniform(0.25, 2.25)
            ph = -2.6 * rr + branch + local.normal(0, 0.105)
            # Convert from galaxy coordinates back to canvas coordinates.
            gx = rr * scale * math.cos(ph)
            gy = rr * scale * math.sin(ph)
            wx = cx + gx * math.cos(theta) + gy * math.sin(theta)
            wy = cy - gx * math.sin(theta) + gy * math.cos(theta)
            amp = local.uniform(0.10, 0.36) * math.exp(-0.18*rr)
            sz = local.uniform(1.0, 3.3)
            if 0 <= wx < width and 0 <= wy < height:
                add_gaussian(knots_blue, wx, wy, amp, sz)
                if local.random() < 0.45:
                    add_gaussian(knots_red, wx + local.normal(0, 2), wy + local.normal(0, 2), amp*0.55, sz*1.4)

    # Foreground stars and background.
    stars = make_starfield(width, height, seed+5)
    bg_gradient = 0.014 + 0.030*(1 - y/height) + 0.010*np.sin(2*np.pi*x/width)
    nebula_wisp = 0.030 * smooth_noise(width, height, seed+22, low=50, scale=1.0)

    total_luma = main + companion + bridge + stars*0.70 + bg_gradient + nebula_wisp
    norm = robust_scale(total_luma, 0.25, 99.93, gamma=0.76)
    main_norm = robust_scale(main, 0.1, 99.75, gamma=0.74)
    star_norm = robust_scale(stars, 74, 99.98, gamma=0.62)
    knots_b = robust_scale(knots_blue, 35, 99.8, gamma=0.72)
    knots_r = robust_scale(knots_red, 35, 99.85, gamma=0.70)
    comp_norm = robust_scale(companion + bridge*0.8, 0.1, 99.8, gamma=0.78)

    rgb = np.zeros((height, width, 3), dtype=np.float32)
    rgb[..., 0] = 0.48*norm + 0.28*main_norm + 0.32*comp_norm + 0.20*knots_r + 0.10*star_norm
    rgb[..., 1] = 0.50*norm + 0.27*main_norm + 0.26*comp_norm + 0.07*knots_b + 0.12*star_norm
    rgb[..., 2] = 0.62*norm + 0.24*main_norm + 0.08*comp_norm + 0.35*knots_b + 0.14*star_norm

    # Cinematic vignette for a phone screen.
    rr_screen = np.sqrt(((x-width/2)/(width/2))**2 + ((y-height/2)/(height/2))**2)
    rgb *= np.clip(1 - 0.48 * rr_screen**1.55, 0.32, 1.0)[..., None]
    return np.clip(rgb, 0, 1)


def fonts():
    try:
        return {
            'title': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 31),
            'big': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 25),
            'body': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18),
            'small': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 15),
            'mono': ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf', 16),
        }
    except Exception:
        return {'title': None, 'big': None, 'body': None, 'small': None, 'mono': None}


def draw_reticle(draw, cx, cy, r, alpha=105):
    col = (210, 230, 255, alpha)
    draw.ellipse([cx-r, cy-r, cx+r, cy+r], outline=col, width=2)
    draw.line([cx-r-14, cy, cx-r+8, cy], fill=col, width=2)
    draw.line([cx+r-8, cy, cx+r+14, cy], fill=col, width=2)
    draw.line([cx, cy-r-14, cx, cy-r+8], fill=col, width=2)
    draw.line([cx, cy+r-8, cx, cy+r+14], fill=col, width=2)


def add_cosmic_rays(arr, frame_index, progress):
    # Raw frames have occasional cosmic-ray streaks; stacking suppresses them later.
    if progress > 0.42:
        return arr
    local = np.random.default_rng(SEED + frame_index * 19)
    out = arr.copy()
    h, w, _ = out.shape
    n = 2 + int(4*(1-progress))
    im = Image.fromarray((out*255).astype(np.uint8), 'RGB')
    d = ImageDraw.Draw(im, 'RGBA')
    for _ in range(n):
        x = int(local.integers(30, w-30))
        y = int(local.integers(100, h-120))
        length = int(local.integers(18, 65))
        angle = local.uniform(-1.0, 1.0)
        dx = int(math.cos(angle) * length)
        dy = int(math.sin(angle) * length)
        a = int(60 + 95*(1-progress))
        d.line([x, y, x+dx, y+dy], fill=(230, 245, 255, a), width=int(local.integers(1, 3)))
    return np.asarray(im).astype(np.float32)/255.0


FONT_CACHE = fonts()

def add_overlay(frame_rgb, frame_index, total_frames, width=W, height=H):
    im = Image.fromarray(np.clip(frame_rgb * 255, 0, 255).astype(np.uint8), 'RGB')
    draw = ImageDraw.Draw(im, 'RGBA')
    f = FONT_CACHE
    p = frame_index / max(total_frames-1, 1)
    n_exposures = 1 + int(p * 144)
    total_seconds = n_exposures * 25
    snr = math.sqrt(n_exposures) * 7.8
    seeing = 1.85 - 0.42*p + 0.06*math.sin(frame_index*0.22)
    sky = 21.08 + 0.18*math.sin(frame_index*0.09 + 1.0)

    # Header block.
    draw.rounded_rectangle([18, 18, width-18, 126], radius=18, fill=(3, 7, 18, 162), outline=(170,190,220,88), width=1)
    draw.text((34, 34), 'M51 WHIRLPOOL GALAXY', fill=(238, 242, 250, 248), font=f['title'])
    draw.text((36, 72), 'live telescope stack • interacting galaxy + tidal bridge', fill=(191, 206, 225, 232), font=f['body'])
    draw.text((36, 98), 'Synthetic FITS-style exposures for a 9:16 YouTube Short', fill=(170, 190, 214, 220), font=f['small'])

    # Targeting graphics.
    draw.line([width*0.50, height*0.52, width*0.59, height*0.30], fill=(170, 205, 255, 45), width=2)
    draw_reticle(draw, width*0.50, height*0.52, 96 + int(4*math.sin(frame_index*0.08)), alpha=88)
    draw_reticle(draw, width*0.59, height*0.30, 42, alpha=75)

    # Telemetry panel.
    panel_x0, panel_y0, panel_x1, panel_y1 = 24, height-314, width-24, height-132
    draw.rounded_rectangle([panel_x0, panel_y0, panel_x1, panel_y1], radius=16, fill=(4, 9, 20, 165), outline=(175,198,225,95), width=1)
    rows = [
        ('Target', 'M51 / NGC 5194 + NGC 5195'),
        ('Frame', f'{frame_index+1:03d}/{total_frames:03d}'),
        ('Stacked', f'{n_exposures:03d} × 25 s  ({total_seconds/60:4.1f} min)'),
        ('Seeing', f'{seeing:0.2f} arcsec'),
        ('Sky', f'{sky:0.2f} mag/arcsec²'),
        ('SNR', f'{snr:0.1f}'),
    ]
    y0 = panel_y0 + 18
    for k, v in rows:
        draw.text((panel_x0+18, y0), k, fill=(154, 178, 205, 232), font=f['small'])
        draw.text((panel_x0+126, y0), v, fill=(234, 240, 250, 245), font=f['mono'])
        y0 += 26

    # Progress bar.
    bx0, by0, bx1, by1 = 24, height-86, width-24, height-52
    draw.text((24, height-118), 'RAW NOISE  →  STACKED SIGNAL', fill=(224, 233, 245, 236), font=f['body'])
    draw.rounded_rectangle([bx0, by0, bx1, by1], radius=12, fill=(5, 10, 20, 175), outline=(170,190,220,100), width=1)
    draw.rounded_rectangle([bx0+4, by0+4, bx0+4+int((bx1-bx0-8)*p), by1-4], radius=10, fill=(226, 236, 255, 162))

    # Sweep line and final text beat.
    sweep_x = int(width * ((frame_index % FPS) / FPS))
    draw.rectangle([sweep_x, 134, min(width, sweep_x+3), height-130], fill=(210, 230, 255, 42))
    if p > 0.78:
        alpha = int(min(210, (p-0.78)/0.22*210))
        draw.rounded_rectangle([50, 152, width-50, 226], radius=16, fill=(5, 11, 22, 120), outline=(210,230,255,70), width=1)
        draw.text((72, 170), 'TIDAL BRIDGE RESOLVED', fill=(240, 246, 255, alpha), font=f['big'])
        draw.text((74, 202), 'signal emerges as exposure time builds', fill=(192, 208, 228, alpha), font=f['small'])

    return np.asarray(im)


def render_video():
    base = make_whirlpool_base()
    Image.fromarray((base*255).astype(np.uint8)).save(PNG_PATH)
    writer = imageio.get_writer(
        VIDEO_PATH,
        fps=FPS,
        codec='libx264',
        quality=8,
        macro_block_size=16,
        output_params=['-pix_fmt', 'yuv420p', '-crf', '23', '-movflags', '+faststart']
    )
    try:
        for i in range(NFRAMES):
            p = i / max(NFRAMES-1, 1)
            nstack = 1 + int(p * 144)
            noise_sigma = 0.095 / math.sqrt(nstack) + 0.0045
            frame = np.clip(base + rng.normal(0, noise_sigma, base.shape).astype(np.float32), 0, 1)
            frame = np.clip((frame - 0.018) * (0.86 + 0.22*p), 0, 1)
            frame = add_cosmic_rays(frame, i, p)
            img = Image.fromarray((frame*255).astype(np.uint8), 'RGB')
            blur_radius = max(0.0, 0.95*(1-p)**1.8)
            if blur_radius > 0.03:
                img = img.filter(ImageFilter.GaussianBlur(radius=blur_radius))

            zoom = 1.00 + 0.105*p + 0.012*math.sin(2*math.pi*p*1.5)
            crop_w, crop_h = int(W / zoom), int(H / zoom)
            center_x = W*(0.50 + 0.025*math.sin(2*math.pi*p))
            center_y = H*(0.50 - 0.030*math.cos(2*math.pi*p*0.7))
            left = int(np.clip(center_x - crop_w/2, 0, W-crop_w))
            top = int(np.clip(center_y - crop_h/2, 0, H-crop_h))
            crop = img.crop((left, top, left+crop_w, top+crop_h)).resize((W, H), Image.Resampling.LANCZOS)
            frame = np.asarray(crop).astype(np.float32) / 255.0
            writer.append_data(add_overlay(frame, i, NFRAMES))
    finally:
        writer.close()
    return base



base = render_video()
print('Saved:', PNG_PATH.resolve())
print('Saved:', VIDEO_PATH.resolve())


Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


Saved: /home/jatin/Downloads/whirlpool_galaxy_shorts_preview.png
Saved: /home/jatin/Downloads/whirlpool_galaxy_telescope_live_stack_shorts.mp4


## Adapting this for real telescope data

Replace the synthetic `base + noise` section in `render_video()` with a running stack from aligned calibrated frames. For example, if you have a NumPy array `data_cube` with shape `(number_of_exposures, height, width)`, use:

```python
stack = np.mean(data_cube[:nstack], axis=0)
frame = robust_scale(stack)
frame_rgb = np.dstack([frame, frame, frame])
writer.append_data(add_overlay(frame_rgb, i, NFRAMES))
```

For FITS frames, install Astropy and read each frame with `from astropy.io import fits`.
